# Burn probability data

Downloads the [Open Climate Risk data](https://source.coop/carbonplan/carbonplan-ocr). Retrieval adapted from [their documentation](https://docs.carbonplan.org/ocr/en/latest/how-to/work-with-data.html#raster-xarray).


In [1]:
import icechunk
import xarray as xr

# Configure S3 storage for the Icechunk repository
version = "v1.1.0"
storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix=f"carbonplan/carbonplan-ocr/output/fire-risk/tensor/production/{version}/ocr.icechunk",
    region="us-west-2",
    anonymous=True,
)

# Open the repository
repo = icechunk.Repository.open(storage)

# Create a read-only session on the main branch
session = repo.readonly_session("main")

## Open the dataset


In [2]:
ds = xr.open_dataset(session.store, engine="zarr", chunks={})
ds

<xarray.Dataset> Size: 652GB
Dimensions:        (latitude: 97579, longitude: 208881)
Coordinates:
  * latitude       (latitude) float64 781kB 22.43 22.43 22.43 ... 52.48 52.48
  * longitude      (longitude) float64 2MB -128.4 -128.4 ... -64.05 -64.05
Data variables:
    bp_2011_riley  (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2011        (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_scott      (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2047_riley  (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_2047       (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2047        (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_2011       (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    crps_scott     (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
Attributes:
    version:          1.1.0
    provider:         CarbonPlan
    terms_of_access:  https://docs.carbonplan.org/ocr/en/latest/terms-of-data...
    data_sources:     https://docs.carbonplan.org/ocr/en/latest/reference/dat...
    license_name:     CC-BY-4.0
    license_url:      https://creativecommons.org/licenses/by/4.0/

Getting the ["Annual burn probability for ~2011 climate conditions"](https://docs.carbonplan.org/ocr/en/latest/reference/data-schema.html#core-risk-variables):


In [3]:
burn_prob = ds["bp_2011"]
burn_prob

<xarray.DataArray 'bp_2011' (latitude: 97579, longitude: 208881)> Size: 82GB
dask.array<open_dataset-bp_2011, shape=(97579, 208881), dtype=float32, chunksize=(6000, 4500), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 781kB 22.43 22.43 22.43 ... 52.48 52.48 52.48
  * longitude  (longitude) float64 2MB -128.4 -128.4 -128.4 ... -64.05 -64.05

These are ["gridded geospatial layers stored at 30m resolution"](https://docs.carbonplan.org/ocr/en/latest/reference/data-schema.html#raster-tensor-datasets). Downsample to 1km:


In [4]:
# Approximate 1 km aggregation from the native ~30 m grid
factor = 33
square_size_m = 30 * factor

burn_prob_1km = burn_prob.coarsen(
    latitude=factor,
    longitude=factor,
    boundary="trim",
).mean()

burn_prob_1km

<xarray.DataArray 'bp_2011' (latitude: 2956, longitude: 6329)> Size: 75MB
dask.array<mean_agg-aggregate, shape=(2956, 6329), dtype=float32, chunksize=(181, 136), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 24kB 22.43 22.44 22.45 ... 52.45 52.46 52.47
  * longitude  (longitude) float64 51kB -128.4 -128.4 -128.4 ... -64.08 -64.07

Store in Parquet for reading by DuckDB:


In [5]:
from pathlib import Path

import dask.dataframe

PARQUET_DIR = Path("data/burn_prob_1km")

ddf: dask.dataframe.DataFrame = burn_prob_1km.to_dask_dataframe()
ddf.to_parquet(PARQUET_DIR, write_index=False)

Number of rows:


In [6]:
ddf.shape[0].compute()

18708524